# 🖼️ CIFAR-10 Image Classification: ANN vs CNN
## Architecture Comparison & Training Strategy Analysis

This notebook covers:
1. **Data Loading & EDA** — CIFAR-10 dataset exploration
2. **ANN Models** — Baseline MLP, Deep MLP, MLP with BatchNorm+Dropout
3. **CNN Models** — Simple CNN, Deep CNN, CNN with advanced regularization
4. **Training Strategies** — SGD vs Adam, LR schedulers, data augmentation
5. **Performance Analysis** — Accuracy, confusion matrices, training curves, comparison tables

## 📦 1. Imports & Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, OneCycleLR

from sklearn.metrics import confusion_matrix, classification_report

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔧 Using device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
NUM_CLASSES = 10

## 📊 2. Data Loading & Exploratory Data Analysis

In [ ]:
# ── Base transforms (no augmentation) ──────────────────────────────────────
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

# ── Augmented transforms (for advanced CNN) ─────────────────────────────────
aug_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

# ── Download datasets ───────────────────────────────────────────────────────
train_base   = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=base_transform)
train_aug    = torchvision.datasets.CIFAR10(root='./data', train=True,  download=False, transform=aug_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=base_transform)

# DataLoaders
BATCH_SIZE = 128
train_loader_base = DataLoader(train_base, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
train_loader_aug  = DataLoader(train_aug,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader       = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples : {len(train_base):,}')
print(f'Test  samples : {len(test_dataset):,}')
print(f'Classes       : {CLASSES}')

In [ ]:
# ── EDA: Sample images ──────────────────────────────────────────────────────
raw_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=False,
                                          transform=transforms.ToTensor())

fig, axes = plt.subplots(4, 10, figsize=(18, 8))
fig.suptitle('CIFAR-10 — Sample Images (4 per class shown)', fontsize=14, fontweight='bold')

class_counts = {i: 0 for i in range(10)}
class_images = {i: [] for i in range(10)}

for img, lbl in raw_train:
    if class_counts[lbl] < 4:
        class_images[lbl].append(img.permute(1,2,0).numpy())
        class_counts[lbl] += 1
    if all(v == 4 for v in class_counts.values()):
        break

for row in range(4):
    for col in range(10):
        ax = axes[row, col]
        ax.imshow(class_images[col][row])
        ax.axis('off')
        if row == 0:
            ax.set_title(CLASSES[col], fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── EDA: Class distribution ──────────────────────────────────────────────────
labels_all = [label for _, label in train_base]
counts = np.bincount(labels_all)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

colors = plt.cm.tab10(np.linspace(0, 1, 10))
bars = ax1.bar(CLASSES, counts, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title('Class Distribution — Train Set', fontweight='bold')
ax1.set_ylabel('Count')
ax1.set_ylim(0, 6500)
for bar, c in zip(bars, counts):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, str(c),
             ha='center', va='bottom', fontsize=8)
ax1.tick_params(axis='x', rotation=30)

ax2.pie(counts, labels=CLASSES, colors=colors, autopct='%1.0f%%', startangle=90)
ax2.set_title('Class Proportions', fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ Dataset is perfectly balanced (5,000 samples per class)')

In [ ]:
# ── EDA: Pixel statistics ────────────────────────────────────────────────────
imgs = torch.stack([raw_train[i][0] for i in range(1000)])  # index individually, slicing not supported
print('Image shape (C×H×W):', imgs[0].shape)
print(f'Pixel mean per channel: R={imgs[:,0].mean():.3f}  G={imgs[:,1].mean():.3f}  B={imgs[:,2].mean():.3f}')
print(f'Pixel std  per channel: R={imgs[:,0].std():.3f}  G={imgs[:,1].std():.3f}  B={imgs[:,2].std():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
ch_names = ['Red', 'Green', 'Blue']
ch_colors = ['red', 'green', 'blue']
for i, (ax, ch, col) in enumerate(zip(axes, ch_names, ch_colors)):
    pixel_vals = imgs[:, i].flatten().numpy()
    ax.hist(pixel_vals, bins=50, color=col, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(f'{ch} Channel Distribution')
    ax.set_xlabel('Pixel value')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 🧠 3. ANN (Artificial Neural Network) Models
### 3A. Architecture Definitions

In [ ]:
# ── ANN-1: Baseline MLP ───────────────────────────────────────────────────────
class BaselineMLP(nn.Module):
    """Simple 2-hidden-layer MLP. Treats image as flat vector."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)


# ── ANN-2: Deep MLP ───────────────────────────────────────────────────────────
class DeepMLP(nn.Module):
    """Deeper MLP with 4 hidden layers."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)


# ── ANN-3: Regularized MLP (BatchNorm + Dropout) ─────────────────────────────
class RegularizedMLP(nn.Module):
    """Deep MLP with BatchNorm and Dropout for better generalization."""
    def __init__(self, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout/2),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('ANN Model Parameter Counts:')
print(f'  BaselineMLP    : {count_params(BaselineMLP()):>10,}')
print(f'  DeepMLP        : {count_params(DeepMLP()):>10,}')
print(f'  RegularizedMLP : {count_params(RegularizedMLP()):>10,}')

### 3B. CNN Architecture Definitions

In [ ]:
# ── CNN-1: Simple CNN ─────────────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    """2-layer conv → 1 FC head."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 512), nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x): return self.classifier(self.features(x))


# ── CNN-2: Deep CNN (VGG-style blocks) ───────────────────────────────────────
class DeepCNN(nn.Module):
    """VGG-style deep CNN with 3 conv blocks."""
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),  nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)


# ── CNN-3: Advanced CNN (BN + Dropout + Residual-like skip) ──────────────────
class ResBlock(nn.Module):
    """Basic residual block with BatchNorm."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)   # skip connection

class AdvancedCNN(nn.Module):
    """CNN with BatchNorm, Dropout, and residual blocks."""
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.layer1 = nn.Sequential(ResBlock(64),  nn.MaxPool2d(2))
        self.down1  = nn.Sequential(
            nn.Conv2d(64, 128, 1, bias=False), nn.BatchNorm2d(128)
        )
        self.layer2 = nn.Sequential(ResBlock(128), nn.MaxPool2d(2))
        self.down2  = nn.Sequential(
            nn.Conv2d(128, 256, 1, bias=False), nn.BatchNorm2d(256)
        )
        self.layer3 = nn.Sequential(ResBlock(256), nn.MaxPool2d(2))
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.down1(x)
        x = self.layer2(x)
        x = self.down2(x)
        x = self.layer3(x)
        x = self.pool(x)
        return self.classifier(x)

print('CNN Model Parameter Counts:')
print(f'  SimpleCNN   : {count_params(SimpleCNN()):>10,}')
print(f'  DeepCNN     : {count_params(DeepCNN()):>10,}')
print(f'  AdvancedCNN : {count_params(AdvancedCNN()):>10,}')

## ⚙️ 4. Training Infrastructure

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scheduler=None, use_onecycle=False):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        if use_onecycle and scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, test_loader, optimizer, criterion,
                epochs=30, scheduler=None, use_onecycle=False, name='Model'):
    """Full training loop with history tracking."""
    model.to(device)
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val_acc = 0.
    start = time.time()

    for epoch in range(1, epochs+1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion,
                                      scheduler, use_onecycle)
        vl_loss, vl_acc = eval_epoch(model, test_loader, criterion)

        if scheduler is not None and not use_onecycle:
            scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), f'best_{name}.pt')

        if epoch % 5 == 0 or epoch == 1:
            elapsed = time.time() - start
            print(f'  [{name}] Epoch {epoch:>3}/{epochs} | '
                  f'Train Acc: {tr_acc:.3f} | Val Acc: {vl_acc:.3f} | '
                  f'Val Loss: {vl_loss:.4f} | Elapsed: {elapsed:.0f}s')

    history['best_val_acc'] = best_val_acc
    history['total_time']   = time.time() - start
    history['params']       = count_params(model)
    return history


def get_predictions(model, loader):
    """Collect all predictions and true labels."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


print('✅ Training utilities ready.')

## 🏋️ 5. Training All Models
### 5A. Train ANN Models

In [ ]:
EPOCHS = 30
criterion = nn.CrossEntropyLoss()
all_results = {}

# ── ANN-1: Baseline MLP (Adam) ─────────────────────────────────────────────
print('═'*65)
print('Training ANN-1: Baseline MLP  (Adam, no regularization)')
print('═'*65)
model_ann1 = BaselineMLP()
opt = optim.Adam(model_ann1.parameters(), lr=1e-3)
all_results['ANN-Baseline'] = train_model(
    model_ann1, train_loader_base, test_loader, opt, criterion,
    epochs=EPOCHS, name='ANN-Baseline'
)

In [ ]:
# ── ANN-2: Deep MLP (SGD + Momentum + StepLR) ─────────────────────────────
print('═'*65)
print('Training ANN-2: Deep MLP  (SGD + Momentum + StepLR)')
print('═'*65)
model_ann2 = DeepMLP()
opt = optim.SGD(model_ann2.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
sched = StepLR(opt, step_size=10, gamma=0.3)
all_results['ANN-Deep'] = train_model(
    model_ann2, train_loader_base, test_loader, opt, criterion,
    epochs=EPOCHS, scheduler=sched, name='ANN-Deep'
)

In [ ]:
# ── ANN-3: Regularized MLP (Adam + CosineAnnealing) ───────────────────────
print('═'*65)
print('Training ANN-3: Regularized MLP  (Adam + CosineAnnealingLR)')
print('═'*65)
model_ann3 = RegularizedMLP()
opt = optim.Adam(model_ann3.parameters(), lr=1e-3, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
all_results['ANN-Regularized'] = train_model(
    model_ann3, train_loader_base, test_loader, opt, criterion,
    epochs=EPOCHS, scheduler=sched, name='ANN-Regularized'
)

### 5B. Train CNN Models

In [ ]:
# ── CNN-1: Simple CNN (Adam) ───────────────────────────────────────────────
print('═'*65)
print('Training CNN-1: Simple CNN  (Adam)')
print('═'*65)
model_cnn1 = SimpleCNN()
opt = optim.Adam(model_cnn1.parameters(), lr=1e-3)
all_results['CNN-Simple'] = train_model(
    model_cnn1, train_loader_base, test_loader, opt, criterion,
    epochs=EPOCHS, name='CNN-Simple'
)

In [ ]:
# ── CNN-2: Deep CNN (SGD + Momentum + StepLR) ─────────────────────────────
print('═'*65)
print('Training CNN-2: Deep CNN  (SGD + Momentum + StepLR)')
print('═'*65)
model_cnn2 = DeepCNN()
opt = optim.SGD(model_cnn2.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
sched = StepLR(opt, step_size=10, gamma=0.3)
all_results['CNN-Deep'] = train_model(
    model_cnn2, train_loader_base, test_loader, opt, criterion,
    epochs=EPOCHS, scheduler=sched, name='CNN-Deep'
)

In [ ]:
# ── CNN-3: Advanced CNN (Adam + OneCycleLR + Data Augmentation) ──────────
print('═'*65)
print('Training CNN-3: Advanced CNN  (Adam + OneCycleLR + Augmentation)')
print('═'*65)
model_cnn3 = AdvancedCNN()
opt = optim.Adam(model_cnn3.parameters(), lr=1e-3, weight_decay=1e-4)
sched = OneCycleLR(opt, max_lr=0.01,
                   steps_per_epoch=len(train_loader_aug), epochs=EPOCHS)
all_results['CNN-Advanced'] = train_model(
    model_cnn3, train_loader_aug, test_loader, opt, criterion,
    epochs=EPOCHS, scheduler=sched, use_onecycle=True, name='CNN-Advanced'
)

## 📈 6. Performance Analysis

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
rows = []
for name, h in all_results.items():
    arch  = 'ANN' if name.startswith('ANN') else 'CNN'
    final = h['val_acc'][-1]
    best  = h['best_val_acc']
    rows.append({
        'Model': name,
        'Type': arch,
        'Parameters': f"{h['params']:,}",
        'Final Val Acc': f"{final*100:.2f}%",
        'Best Val Acc': f"{best*100:.2f}%",
        'Train Time (s)': f"{h['total_time']:.0f}s"
    })

df = pd.DataFrame(rows)
print('='*80)
print('RESULTS SUMMARY')
print('='*80)
print(df.to_string(index=False))
print('='*80)

In [ ]:
# ── Training Curves: Loss ────────────────────────────────────────────────────
colors_ann = ['#e74c3c','#c0392b','#922b21']
colors_cnn = ['#2980b9','#1a5276','#148f77']

ann_models = [(k, v) for k, v in all_results.items() if k.startswith('ANN')]
cnn_models = [(k, v) for k, v in all_results.items() if k.startswith('CNN')]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training Curves: ANN vs CNN', fontsize=15, fontweight='bold')

epochs_range = range(1, EPOCHS+1)

# ANN Loss
ax = axes[0,0]
for (name, h), col in zip(ann_models, colors_ann):
    ax.plot(epochs_range, h['train_loss'], '--', color=col, alpha=0.6, label=f'{name} (train)')
    ax.plot(epochs_range, h['val_loss'],   '-',  color=col, label=f'{name} (val)')
ax.set_title('ANN — Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# CNN Loss
ax = axes[0,1]
for (name, h), col in zip(cnn_models, colors_cnn):
    ax.plot(epochs_range, h['train_loss'], '--', color=col, alpha=0.6, label=f'{name} (train)')
    ax.plot(epochs_range, h['val_loss'],   '-',  color=col, label=f'{name} (val)')
ax.set_title('CNN — Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ANN Accuracy
ax = axes[1,0]
for (name, h), col in zip(ann_models, colors_ann):
    ax.plot(epochs_range, [a*100 for a in h['train_acc']], '--', color=col, alpha=0.6, label=f'{name} (train)')
    ax.plot(epochs_range, [a*100 for a in h['val_acc']],   '-',  color=col, label=f'{name} (val)')
ax.set_title('ANN — Accuracy', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# CNN Accuracy
ax = axes[1,1]
for (name, h), col in zip(cnn_models, colors_cnn):
    ax.plot(epochs_range, [a*100 for a in h['train_acc']], '--', color=col, alpha=0.6, label=f'{name} (train)')
    ax.plot(epochs_range, [a*100 for a in h['val_acc']],   '-',  color=col, label=f'{name} (val)')
ax.set_title('CNN — Accuracy', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart: Best Validation Accuracy ──────────────────────────────────────
model_names = list(all_results.keys())
best_accs   = [h['best_val_acc']*100 for h in all_results.values()]
bar_colors  = ['#e74c3c','#c0392b','#922b21','#2980b9','#1a5276','#148f77']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(model_names, best_accs, color=bar_colors, edgecolor='black', linewidth=0.7)
ax.set_title('Best Validation Accuracy — All Models', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='Random baseline (10%)')

for bar, acc in zip(bars, best_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Annotate ANN vs CNN regions
ax.axvspan(-0.5, 2.5, alpha=0.07, color='red',  label='ANN region')
ax.axvspan(2.5,  5.5, alpha=0.07, color='blue', label='CNN region')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Parameters vs Accuracy scatter ───────────────────────────────────────────
params_list = [h['params'] for h in all_results.values()]
best_accs   = [h['best_val_acc']*100 for h in all_results.values()]
times_list  = [h['total_time'] for h in all_results.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Params vs Accuracy
scatter_colors = ['#e74c3c','#c0392b','#922b21','#2980b9','#1a5276','#148f77']
for name, p, a, col in zip(model_names, params_list, best_accs, scatter_colors):
    ax1.scatter(p, a, s=120, color=col, zorder=5, edgecolors='black')
    ax1.annotate(name, (p, a), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax1.set_title('Parameters vs Best Accuracy', fontweight='bold')
ax1.set_xlabel('# Parameters')
ax1.set_ylabel('Accuracy (%)')
ax1.grid(True, alpha=0.3)

# Training time vs Accuracy
for name, t, a, col in zip(model_names, times_list, best_accs, scatter_colors):
    ax2.scatter(t, a, s=120, color=col, zorder=5, edgecolors='black')
    ax2.annotate(name, (t, a), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax2.set_title('Training Time vs Best Accuracy', fontweight='bold')
ax2.set_xlabel('Training Time (s)')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Overfitting analysis: train-val gap ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Overfitting Analysis: Train vs Val Accuracy Gap', fontweight='bold')

for ax, models, title in [(axes[0], ann_models, 'ANN Models'),
                           (axes[1], cnn_models, 'CNN Models')]:
    cols = colors_ann if 'ANN' in title else colors_cnn
    for (name, h), col in zip(models, cols):
        gap = [t*100 - v*100 for t, v in zip(h['train_acc'], h['val_acc'])]
        ax.plot(epochs_range, gap, color=col, label=name)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.fill_between(epochs_range, 0, 30, alpha=0.05, color='red',
                    label='Overfitting zone')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Acc – Val Acc (%)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-5, 40)

plt.tight_layout()
plt.show()

## 🎯 7. Confusion Matrices & Classification Reports

In [ ]:
# Load best weights and get predictions
model_objects = {
    'ANN-Baseline':   model_ann1,
    'ANN-Deep':       model_ann2,
    'ANN-Regularized':model_ann3,
    'CNN-Simple':     model_cnn1,
    'CNN-Deep':       model_cnn2,
    'CNN-Advanced':   model_cnn3,
}

predictions = {}
for name, model in model_objects.items():
    try:
        model.load_state_dict(torch.load(f'best_{name}.pt', map_location=device))
    except:
        pass
    preds, labels = get_predictions(model, test_loader)
    predictions[name] = (preds, labels)

print('✅ Predictions collected for all models.')

In [ ]:
# ── Confusion Matrices (2x3 grid) ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, (name, (preds, labels)) in zip(axes.flat, predictions.items()):
    cm   = confusion_matrix(labels, preds)
    acc  = np.trace(cm) / cm.sum()
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalized

    cmap = 'Reds' if name.startswith('ANN') else 'Blues'
    sns.heatmap(cm_n, annot=cm, fmt='d', cmap=cmap, ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.5, cbar=False,
                annot_kws={'size': 7})
    ax.set_title(f'{name}\n(Test Acc: {acc*100:.1f}%)', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Per-class F1 scores for best ANN vs best CNN ─────────────────────────────
from sklearn.metrics import f1_score

# Find best ANN and best CNN
ann_names = [k for k in all_results if k.startswith('ANN')]
cnn_names = [k for k in all_results if k.startswith('CNN')]

best_ann = max(ann_names, key=lambda k: all_results[k]['best_val_acc'])
best_cnn = max(cnn_names, key=lambda k: all_results[k]['best_val_acc'])

f1_ann = f1_score(predictions[best_ann][1], predictions[best_ann][0], average=None)
f1_cnn = f1_score(predictions[best_cnn][1], predictions[best_cnn][0], average=None)

x = np.arange(NUM_CLASSES)
width = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, f1_ann, width, label=f'Best ANN ({best_ann})', color='#c0392b', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_cnn, width, label=f'Best CNN ({best_cnn})', color='#1a5276', alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=25)
ax.set_ylabel('F1-Score'); ax.set_ylim(0, 1.05)
ax.set_title('Per-Class F1 Score: Best ANN vs Best CNN', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Full classification report for best models ────────────────────────────────
print('='*60)
print(f'Classification Report — {best_ann}')
print('='*60)
print(classification_report(predictions[best_ann][1], predictions[best_ann][0],
                             target_names=CLASSES))

print('\n'+'='*60)
print(f'Classification Report — {best_cnn}')
print('='*60)
print(classification_report(predictions[best_cnn][1], predictions[best_cnn][0],
                             target_names=CLASSES))

## 🔍 8. Error Analysis — Misclassified Examples

In [ ]:
def show_misclassified(model, loader, name, n=20):
    model.eval()
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
    std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3,1,1)

    wrong_imgs, wrong_preds, wrong_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            out   = model(imgs.to(device)).argmax(1).cpu()
            mask  = (out != labels)
            wrong_imgs.extend(imgs[mask])
            wrong_preds.extend(out[mask].tolist())
            wrong_labels.extend(labels[mask].tolist())
            if len(wrong_imgs) >= n: break

    cols = min(n, 10)
    rows = (min(n, len(wrong_imgs)) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.6, rows*1.8))
    fig.suptitle(f'Misclassified Samples — {name}', fontsize=11, fontweight='bold')
    for i, ax in enumerate(np.array(axes).flat):
        if i >= len(wrong_imgs):
            ax.axis('off'); continue
        img = wrong_imgs[i] * std + mean
        img = img.permute(1,2,0).clamp(0,1).numpy()
        ax.imshow(img)
        ax.set_title(f'T:{CLASSES[wrong_labels[i]][:4]}\nP:{CLASSES[wrong_preds[i]][:4]}',
                     fontsize=7, color='red')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_misclassified(model_objects[best_ann], test_loader, best_ann, n=20)
show_misclassified(model_objects[best_cnn], test_loader, best_cnn, n=20)

## 🔬 9. Training Strategy Comparison

In [ ]:
# ── Strategy comparison: Same architecture (SimpleCNN), different optimizers ──
print('Comparing Training Strategies on SimpleCNN...')
strategy_results = {}
EPOCHS_STRAT = 20

strategies = [
    ('SGD (no momentum)', lambda m: optim.SGD(m.parameters(), lr=0.01)),
    ('SGD + Momentum',    lambda m: optim.SGD(m.parameters(), lr=0.01, momentum=0.9)),
    ('RMSProp',           lambda m: optim.RMSprop(m.parameters(), lr=1e-3)),
    ('Adam',              lambda m: optim.Adam(m.parameters(), lr=1e-3)),
    ('AdamW',             lambda m: optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2)),
]

for strat_name, opt_fn in strategies:
    print(f'  → {strat_name}')
    m = SimpleCNN().to(device)
    opt = opt_fn(m)
    h = train_model(m, train_loader_base, test_loader, opt, criterion,
                    epochs=EPOCHS_STRAT, name=f'strat_{strat_name}')
    strategy_results[strat_name] = h

In [ ]:
# ── Plot optimizer comparison ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimizer Comparison on SimpleCNN', fontweight='bold', fontsize=13)

colors_strat = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db']
er = range(1, EPOCHS_STRAT+1)

for (sn, h), col in zip(strategy_results.items(), colors_strat):
    ax1.plot(er, h['train_loss'], '--', color=col, alpha=0.5)
    ax1.plot(er, h['val_loss'],   '-',  color=col, label=sn)
    ax2.plot(er, [a*100 for a in h['val_acc']], color=col, label=f'{sn} (best:{h["best_val_acc"]*100:.1f}%)')

ax1.set_title('Loss (dashed=train, solid=val)'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── LR Scheduler comparison ───────────────────────────────────────────────────
print('Comparing LR Schedulers on SimpleCNN (Adam base)...')
sched_results = {}

schedules = [
    ('No Scheduler',    None,            False),
    ('StepLR(γ=0.5)',   lambda opt, ldr: StepLR(opt, step_size=5, gamma=0.5),   False),
    ('CosineAnnealing', lambda opt, ldr: CosineAnnealingLR(opt, T_max=EPOCHS_STRAT, eta_min=1e-5), False),
    ('OneCycleLR',      lambda opt, ldr: OneCycleLR(opt, max_lr=0.01,
                                                    steps_per_epoch=len(ldr),
                                                    epochs=EPOCHS_STRAT), True),
]

for sched_name, sched_fn, is_onecycle in schedules:
    print(f'  → {sched_name}')
    m   = SimpleCNN().to(device)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    sch = sched_fn(opt, train_loader_base) if sched_fn else None
    h   = train_model(m, train_loader_base, test_loader, opt, criterion,
                      epochs=EPOCHS_STRAT, scheduler=sch,
                      use_onecycle=is_onecycle, name=f'sched_{sched_name}')
    sched_results[sched_name] = h

In [ ]:
# ── Plot scheduler comparison ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle('LR Scheduler Comparison on SimpleCNN (Adam)', fontweight='bold')

colors_sched = ['#95a5a6','#e74c3c','#3498db','#2ecc71']
for (sn, h), col in zip(sched_results.items(), colors_sched):
    ax.plot(range(1, EPOCHS_STRAT+1), [a*100 for a in h['val_acc']],
            color=col, label=f'{sn} (best: {h["best_val_acc"]*100:.1f}%)', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📋 10. Final Summary Dashboard

In [ ]:
# ── Radar chart: Multi-dimensional model comparison ───────────────────────────
import matplotlib.patches as mpatches

# Build comparison metrics for 6 models
metrics_data = []
for name, h in all_results.items():
    preds, labels = predictions[name]
    acc     = np.mean(preds == labels) * 100
    f1_mac  = f1_score(labels, preds, average='macro') * 100
    gap     = (h['train_acc'][-1] - h['val_acc'][-1]) * 100          # overfitting
    efficiency = acc / (h['params'] / 1e6)                            # acc per M params
    speed   = acc / (h['total_time'] / 60)                            # acc per minute
    metrics_data.append({'Model': name, 'Test Acc (%)': round(acc,2),
                          'Macro F1 (%)': round(f1_mac,2),
                          'Overfit Gap (%)': round(gap,2),
                          'Params (M)': round(h['params']/1e6,2),
                          'Time (s)': round(h['total_time'],0)})

df_final = pd.DataFrame(metrics_data)
print('='*75)
print('FINAL MODEL COMPARISON')
print('='*75)
print(df_final.to_string(index=False))
print('='*75)

# Winner
best_model = df_final.loc[df_final['Test Acc (%)'].idxmax(), 'Model']
best_acc   = df_final['Test Acc (%)'].max()
print(f'\n🏆 Best model: {best_model}  ({best_acc:.1f}% test accuracy)')

In [ ]:
# ── Key takeaways ─────────────────────────────────────────────────────────────
ann_best_acc = max(all_results[k]['best_val_acc']*100 for k in ann_names)
cnn_best_acc = max(all_results[k]['best_val_acc']*100 for k in cnn_names)

print('='*65)
print('KEY FINDINGS')
print('='*65)
print(f'1. ANN Best Accuracy : {ann_best_acc:.1f}%')
print(f'   CNN Best Accuracy : {cnn_best_acc:.1f}%')
print(f'   CNN advantage     : +{cnn_best_acc - ann_best_acc:.1f}%')
print()
print('2. WHY CNNs WIN on image data:')
print('   - Exploit spatial locality via convolution kernels')
print('   - Translation invariance from pooling layers')
print('   - Parameter sharing → far fewer weights per feature')
print('   - Hierarchical feature learning (edges → shapes → objects)')
print()
print('3. ANN LIMITATIONS on CIFAR-10:')
print('   - Flatten destroys spatial structure')
print('   - O(N²) fully-connected → massive parameter count')
print('   - High overfitting even with BatchNorm + Dropout')
print()
print('4. TRAINING STRATEGY INSIGHTS:')
print('   - Adam converges faster; SGD+momentum can match with tuning')
print('   - CosineAnnealing / OneCycleLR outperform StepLR in most cases')
print('   - Data augmentation is crucial for preventing CNN overfitting')
print('   - BatchNorm + Dropout together provide strongest regularization')
print('='*65)